# 02 — Baselines: Dictionary and LSTM seq2seq

**`BASE-01` — dictionary / position baseline** *(legacy id: the project's original baseline)*.

A non-neural lower bound: for each Arabic subword, predict the English subword most frequently
seen at the same *relative* position in training, then stitch the guesses together. It has no
learning and no context — it exists only to show how far a trivial lexical method gets, so the
trained Transformer (notebook 03) can be measured against it.

This notebook contains **only** the dictionary baseline — no Transformer results, no beam search.

## Setup

In [1]:
import os
from collections import Counter, defaultdict
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
import pandas as pd, sacrebleu, sentencepiece as spm
TOK, VOC, MAX_LEN = 'Data/tokenized', 'Data/vocab', 80
sp_ar = spm.SentencePieceProcessor(model_file=f'{VOC}/sp_ar.model')
sp_en = spm.SentencePieceProcessor(model_file=f'{VOC}/sp_en.model')
def read_lines(p): return open(p, encoding='utf-8').read().splitlines()

## Load data (same filtered test set as the Transformer)

In [2]:
train_ar, train_en = read_lines(f'{TOK}/train.ar.bpe'), read_lines(f'{TOK}/train.en.bpe')
kept = [(a, e) for a, e in zip(read_lines(f'{TOK}/test.ar.bpe'), read_lines(f'{TOK}/test.en.bpe'))
        if len(a.split()) <= MAX_LEN and len(e.split()) <= MAX_LEN]
test_ar = [a for a, e in kept]; test_en = [e for a, e in kept]
print(f'train {len(train_ar):,} | test kept {len(kept):,}')

train 50,000 | test kept 8,491


## The baseline algorithm
Most-frequent English subword per Arabic subword at the same relative position.

In [3]:
def build_position_dictionary(src_lines, tgt_lines):
    counts = defaultdict(Counter)
    for src, tgt in zip(src_lines, tgt_lines):
        s, t = src.split(), tgt.split()
        if not s or not t:
            continue
        for i, piece in enumerate(s):
            j = min(round(i * (len(t) - 1) / max(1, len(s) - 1)), len(t) - 1)
            counts[piece][t[j]] += 1
    return {p: c.most_common(1)[0][0] for p, c in counts.items()}

def translate_baseline(line, dictionary):
    return ' '.join(dictionary.get(p, '<unk>') for p in line.split())

dictionary = build_position_dictionary(train_ar, train_en)
print('dictionary entries:', f'{len(dictionary):,}')
list(dictionary.items())[:8]

dictionary entries: 8,173


[('▁اذاً', '▁so'),
 ('▁نحن', '▁we'),
 ('▁ن', '▁we'),
 ('بي', ','),
 ('عها', '▁to'),
 ('،', ','),
 ('▁ثم', '▁and'),
 ('▁شئ', '▁something')]

## Scoring (BLEU + chrF++ on detokenized English)

In [4]:
def detok(lines, sp): return [sp.decode(l.split()) for l in lines]
def score(hyp, ref):
    return (round(sacrebleu.corpus_bleu(hyp, [ref]).score, 4),
            round(sacrebleu.corpus_chrf(hyp, [ref], word_order=2).score, 4))

baseline = detok([translate_baseline(l, dictionary) for l in test_ar], sp_en)
reference = detok(test_en, sp_en)
bleu_full, chrf_full = score(baseline, reference)
bleu_1k, chrf_1k = score(baseline[:1000], reference[:1000])
results = pd.DataFrame([
    {'local_id': 'BASE-01', 'old_id': 'dictionary_position_baseline', 'split': 'test_full',
     'examples': len(baseline), 'bleu': bleu_full, 'chrf_pp': chrf_full},
    {'local_id': 'BASE-01', 'old_id': 'dictionary_position_baseline', 'split': 'test_1000',
     'examples': 1000, 'bleu': bleu_1k, 'chrf_pp': chrf_1k},
])
results

,local_id,old_id,split,examples,bleu,chrf_pp
0,BASE-01,dictionary_position_baseline,test_full,8491,4.7777,25.6503
1,BASE-01,dictionary_position_baseline,test_1000,1000,4.7137,25.5561


In [5]:
os.makedirs('outputs/tables', exist_ok=True)
pd.DataFrame([{'model': 'dictionary_position_baseline', 'local_id': 'BASE-01', 'decoding': 'position_dictionary',
               'examples': len(baseline), 'bleu': bleu_full, 'chrf_pp': chrf_full}]).to_csv(
    'outputs/tables/baseline_results.csv', index=False)
print('saved outputs/tables/baseline_results.csv')

saved outputs/tables/baseline_results.csv


## Qualitative examples
Word-salad: correct vocabulary, no grammar — exactly what a floor should look like.

In [6]:
samples = pd.DataFrame({'source_ar': detok(test_ar[:6], sp_ar), 'reference_en': reference[:6],
                        'baseline_output': baseline[:6]})
os.makedirs('outputs/examples', exist_ok=True)
samples.to_csv('outputs/examples/baseline_samples.csv', index=False, encoding='utf-8')
samples

,source_ar,reference_en,baseline_output
0,قبل عدة سنوات، هنا في تيد، قدّم بيتر سكيلمان م...,"several years ago here at ted, peter skillman ...","before many years, here in ted,,, peter, form,..."
1,والفكرة غاية في البساطة. فريق مكوّن من اربعة ي...,and the idea's pretty simple: teams of four ha...,"and the very in simplicity. a, a of four we to..."
2,يجب ان تكون المارش مالو علي القمة.,the marshmallow has to be on top.,"we to be the the, money, on the."
3,ورغماً عن انها تبدو بسيطة للغاية، الا انها صعب...,"and, though it seems really simple, it's actua...","and the, about it look simple., the it difficu..."
4,لذا فقد فكرت بان هذه فكرة مثيرة، وقمت بتضمينها...,"and so, i thought this was an interesting idea...","so we i that this idea interesting, and to,. i..."
5,وقد كان نجاحاً باهراً.,and it was a huge success.,"and was success, the,."


## Limitations of the dictionary baseline
No context, no reordering, no fluency — it cannot model agreement, word order, or multi-word
expressions. Its BLEU is very low; interestingly its **chrF++** is competitive (it emits many
correct content subwords). It is a lower bound only — the neural model is in notebook 03.

## Neural baseline: LSTM seq2seq\nThe dictionary baseline above is non-neural. As a stronger, *learned* lower bound we also train a small LSTM sequence-to-sequence model: a bidirectional LSTM encoder, a single-layer LSTM decoder with Luong-general attention, and the output projection tied to the target embedding. It is trained from scratch on the same BPE data as the Tiny Transformer, so the comparison isolates *recurrent vs. self-attention* at a comparable parameter budget. Following the same CPU-by-design convention as the training notebook, we define the model and its training loop in full but **load the saved checkpoint** here; set `TRAIN_LSTM = True` to retrain from scratch.

In [7]:
import os, random
import torch
from torch import nn
import torch.nn.functional as F
torch.manual_seed(42)

# token vocabularies, read from the same SentencePiece models used everywhere else
ar_tokens = [l.split("\t")[0] for l in read_lines(f"{VOC}/sp_ar.vocab") if l]
en_tokens = [l.split("\t")[0] for l in read_lines(f"{VOC}/sp_en.vocab") if l]
ar_stoi = {t: i for i, t in enumerate(ar_tokens)}
en_stoi = {t: i for i, t in enumerate(en_tokens)}
en_itos = {i: t for i, t in enumerate(en_tokens)}
ar_pad, en_pad = ar_stoi["<pad>"], en_stoi["<pad>"]
en_bos, en_eos = en_stoi["<s>"], en_stoi["</s>"]
EMB, HID = 128, 256


class LSTMSeq2Seq(nn.Module):
    """Bi-LSTM encoder + single-layer LSTM decoder with Luong-general attention; output tied to tgt embedding."""
    def __init__(self):
        super().__init__()
        self.src_emb = nn.Embedding(len(ar_tokens), EMB, padding_idx=ar_pad)
        self.tgt_emb = nn.Embedding(len(en_tokens), EMB, padding_idx=en_pad)
        self.encoder = nn.LSTM(EMB, HID, batch_first=True, bidirectional=True)
        self.bridge_h = nn.Linear(2 * HID, HID)
        self.bridge_c = nn.Linear(2 * HID, HID)
        self.decoder = nn.LSTM(EMB, HID, batch_first=True)
        self.attn = nn.Linear(HID, 2 * HID, bias=False)            # Luong general: dec . W . enc
        self.combine = nn.Linear(HID + 2 * HID, EMB)               # -> EMB so output can tie with tgt_emb
        self.out = nn.Linear(EMB, len(en_tokens))
        self.out.weight = self.tgt_emb.weight

    def encode(self, src):
        eo, (h, c) = self.encoder(self.src_emb(src))               # eo [B,T,2H]; h,c [2,B,H]
        h0 = torch.tanh(self.bridge_h(torch.cat([h[0], h[1]], -1))).unsqueeze(0)
        c0 = torch.tanh(self.bridge_c(torch.cat([c[0], c[1]], -1))).unsqueeze(0)
        return eo, (h0, c0)

    def attend(self, dec_out, eo, src_mask):
        scores = torch.bmm(self.attn(dec_out), eo.transpose(1, 2))  # [B,Tt,Ts]
        scores = scores.masked_fill(src_mask.unsqueeze(1), float("-inf"))
        ctx = torch.bmm(torch.softmax(scores, -1), eo)              # [B,Tt,2H]
        return self.out(torch.tanh(self.combine(torch.cat([dec_out, ctx], -1))))

    def forward(self, src, tgt_in):
        eo, st = self.encode(src)
        dec_out, _ = self.decoder(self.tgt_emb(tgt_in), st)
        return self.attend(dec_out, eo, src.eq(ar_pad))


def encode_ids(tokens, stoi):
    return [stoi["<s>"]] + [stoi.get(x, stoi["<unk>"]) for x in tokens] + [stoi["</s>"]]

The training loop (Adam, gradient clipping, best-checkpoint selection on validation loss, 60 epochs) is shown for completeness and gated behind `TRAIN_LSTM`. By default we load the saved best checkpoint.

In [8]:
CKPT = "outputs/checkpoints/baselines/lstm_seq2seq/best.pt"
TRAIN_LSTM = False          # CPU-by-design: load the trained checkpoint; set True to retrain (~GPU)


def pad_batch(chunk):
    sm = max(len(s) for s, t in chunk); tm = max(len(t) for s, t in chunk)
    src = torch.full((len(chunk), sm), ar_pad); tgt = torch.full((len(chunk), tm), en_pad)
    for r, (s, t) in enumerate(chunk):
        src[r, :len(s)] = torch.tensor(s); tgt[r, :len(t)] = torch.tensor(t)
    return src, tgt


def train_lstm(model, epochs=60):
    random.seed(42); torch.manual_seed(42)
    def load_pairs(a_path, e_path):
        return [(encode_ids(a.split(), ar_stoi), encode_ids(e.split(), en_stoi))
                for a, e in zip(read_lines(a_path), read_lines(e_path))
                if len(a.split()) <= MAX_LEN and len(e.split()) <= MAX_LEN]
    train = load_pairs(f"{TOK}/train.ar.bpe", f"{TOK}/train.en.bpe")
    val = load_pairs(f"{TOK}/validation.ar.bpe", f"{TOK}/validation.en.bpe")
    opt = torch.optim.Adam(model.parameters(), lr=3e-4)

    @torch.no_grad()
    def val_loss():
        model.eval(); tot = n = 0
        for i in range(0, len(val), 32):
            src, tgt = pad_batch(val[i:i + 32]); gold = tgt[:, 1:].reshape(-1)
            ce = F.cross_entropy(model(src, tgt[:, :-1]).reshape(-1, len(en_tokens)), gold, ignore_index=en_pad)
            k = gold.ne(en_pad).sum().item(); tot += ce.item() * k; n += k
        return tot / max(1, n)

    os.makedirs(os.path.dirname(CKPT), exist_ok=True)
    best, hist = float("inf"), []
    for ep in range(1, epochs + 1):
        model.train(); order = list(range(len(train))); random.shuffle(order)
        for i in range(0, len(order), 32):
            src, tgt = pad_batch([train[j] for j in order[i:i + 32]]); gold = tgt[:, 1:].reshape(-1)
            loss = F.cross_entropy(model(src, tgt[:, :-1]).reshape(-1, len(en_tokens)), gold, ignore_index=en_pad)
            opt.zero_grad(set_to_none=True); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        v = val_loss(); hist.append({"epoch": ep, "val_loss": round(v, 4)})
        if v <= best:
            best = v
            torch.save({"model_state_dict": model.state_dict(), "best_val_loss": best, "history": hist}, CKPT)
    return model


lstm = LSTMSeq2Seq()
if TRAIN_LSTM or not os.path.exists(CKPT):
    lstm = train_lstm(lstm)
else:
    print("loading saved checkpoint (set TRAIN_LSTM=True to retrain)")
lstm.load_state_dict(torch.load(CKPT, map_location="cpu")["model_state_dict"])
lstm.eval()
lstm_best_val = round(torch.load(CKPT, map_location="cpu")["best_val_loss"], 4)
print("LSTM parameters:", f"{sum(p.numel() for p in lstm.parameters()):,}", "| best val loss:", lstm_best_val)

loading saved checkpoint (set TRAIN_LSTM=True to retrain)


LSTM parameters: 3,733,952 | best val loss: 4.1219


We greedy-decode the **same 1000-sentence test slice** used for the ablation subset scores and evaluate with the same BLEU / chrF++, plus the repetition diagnostic used throughout.

In [9]:
@torch.no_grad()
def lstm_translate(src_ids, maxlen=80):
    src = torch.tensor([src_ids]); eo, st = lstm.encode(src); mask = src.eq(ar_pad)
    ys = torch.tensor([[en_bos]]); produced = []
    for _ in range(maxlen):
        dec_out, st = lstm.decoder(lstm.tgt_emb(ys[:, -1:]), st)
        nxt = int(lstm.attend(dec_out, eo, mask)[:, -1].argmax(-1))
        if nxt == en_eos:
            break
        produced.append(nxt); ys = torch.cat([ys, torch.tensor([[nxt]])], 1)
    return sp_en.decode([en_itos.get(i, "<unk>") for i in produced])


def repetition_rate(hyps):
    def degenerate(h):
        w = h.split()
        if len(w) < 4:
            return False
        bg = list(zip(w, w[1:]))
        return (len(bg) - len(set(bg))) >= 2 or len(set(w)) / len(w) < 0.5
    return round(100 * sum(degenerate(h) for h in hyps) / max(1, len(hyps)), 1)


# same 1000-sentence test slice used for the ablation subset scores
lstm_hyps = [lstm_translate(encode_ids(a.split(), ar_stoi)) for a in test_ar[:1000]]
lstm_bleu, lstm_chrf = score(lstm_hyps, reference[:1000])
lstm_rep = repetition_rate(lstm_hyps)
print(f"LSTM seq2seq (greedy, 1000): BLEU {lstm_bleu} | chrF++ {lstm_chrf} | repetition {lstm_rep}%")

LSTM seq2seq (greedy, 1000): BLEU 9.5587 | chrF++ 30.328 | repetition 28.3%


Saving the result and placing the two baselines side by side on the 1000-sentence slice.

In [10]:
os.makedirs("outputs/tables/baselines", exist_ok=True)
pd.DataFrame([{"experiment": "LSTM seq2seq (greedy)", "category": "Baselines",
               "bleu": lstm_bleu, "chrf_pp": lstm_chrf, "rep_pct": lstm_rep,
               "best_val_loss": lstm_best_val}]).to_csv(
    "outputs/tables/baselines/lstm_seq2seq_result.csv", index=False)

baseline_compare = pd.DataFrame([
    {"baseline": "dictionary (position)", "bleu": bleu_1k, "chrf_pp": chrf_1k},
    {"baseline": "LSTM seq2seq (greedy)", "bleu": lstm_bleu, "chrf_pp": lstm_chrf},
])
baseline_compare

,baseline,bleu,chrf_pp
0,dictionary (position),4.7137,25.5561
1,LSTM seq2seq (greedy),9.5587,30.3280


A few LSTM outputs against the reference.

In [11]:
pd.DataFrame({"source_ar": detok(test_ar[:6], sp_ar),
              "reference_en": reference[:6],
              "lstm_en": lstm_hyps[:6]})

,source_ar,reference_en,lstm_en
0,قبل عدة سنوات، هنا في تيد، قدّم بيتر سكيلمان م...,"several years ago here at ted, peter skillman ...","a few years ago, here at ted, we might have a ..."
1,والفكرة غاية في البساطة. فريق مكوّن من اربعة ي...,and the idea's pretty simple: teams of four ha...,and the idea of the idea of the general is a l...
2,يجب ان تكون المارش مالو علي القمة.,the marshmallow has to be on top.,the own girl's on the top of the top.
3,ورغماً عن انها تبدو بسيطة للغاية، الا انها صعب...,"and, though it seems really simple, it's actua...",and even a very simple thing that it's very si...
4,لذا فقد فكرت بان هذه فكرة مثيرة، وقمت بتضمينها...,"and so, i thought this was an interesting idea...","so i thought that this idea, i've got to be an..."
5,وقد كان نجاحاً باهراً.,and it was a huge success.,it was a huge amount.


The LSTM is a genuinely competitive baseline: at **BLEU 9.55 / chrF++ 30.32** it clears the dictionary baseline by a wide margin and even edges past the *un-regularized* from-scratch Transformer baseline (7.99 BLEU, greedy) reported in the next notebook. Its high repetition rate (~28\%) is the same degeneracy the decoding study later controls. This sets the bar correctly: the regularized Transformer with tuned decoding (final 19.91 BLEU) must beat this learned LSTM baseline, not merely the trivial dictionary, to justify the self-attention architecture.